# Experiment 9: Creating REST API for Model Deployment
- Designing a RESTful API to serve machine learning models using Flask
- Containerization with Docker: Containerizing the model and API using Docker

*Note: This notebook generates the application code and Dockerfile required for the deployment, and tests the local Flask server.*

## 1. Train and Save a Simple Model

In [1]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import joblib
import os

os.makedirs('app', exist_ok=True)

# Train a model
iris = load_iris()
clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(iris.data, iris.target)

# Save the model
joblib.dump(clf, 'app/model.pkl')
print("Model saved to app/model.pkl")

Model saved to app/model.pkl


## 2. Create the Flask Application

In [2]:
flask_app_code = '''from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)
model = joblib.load('model.pkl')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.json
        features = np.array(data['features']).reshape(1, -1)
        prediction = model.predict(features)
        return jsonify({'prediction': int(prediction[0])})
    except Exception as e:
        return jsonify({'error': str(e)})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
'''

with open('app/app.py', 'w') as f:
    f.write(flask_app_code)
print("Flask app generated at app/app.py")

Flask app generated at app/app.py


## 3. Create requirements.txt and Dockerfile

In [3]:
requirements = '''Flask==3.0.3
joblib==1.4.2
scikit-learn==1.5.0
numpy==1.26.4
werkzeug==3.0.3
'''

with open('app/requirements.txt', 'w') as f:
    f.write(requirements)

dockerfile = '''FROM python:3.9-slim

WORKDIR /app

COPY requirements.txt requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 5000

CMD ["python", "app.py"]
'''

with open('app/Dockerfile', 'w') as f:
    f.write(dockerfile)

print("Dockerfile and requirements.txt generated.")

Dockerfile and requirements.txt generated.


## 4. Test the API Locally
We will briefly start the Flask app in a background process, send a request, and terminate it.

In [4]:
import subprocess
import time
import requests
import sys

# Start the Flask app
flask_process = subprocess.Popen([sys.executable, 'app.py'], cwd='app')

# Give it a second to start
time.sleep(2)

try:
    # Send a test request
    test_data = {'features': [5.1, 3.5, 1.4, 0.2]}
    response = requests.post('http://127.0.0.1:5000/predict', json=test_data)
    print(f"API Response: {response.json()}")
finally:
    # Terminate the process
    flask_process.terminate()
    flask_process.wait()
    print("Flask server terminated.")

/Users/mukilan/Build-and-Deploy-Machine-Learning/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


 * Serving Flask app 'app'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.31.172:5000
Press CTRL+C to quit


API Response: {'prediction': 0}
Flask server terminated.


127.0.0.1 - - [28/Sep/2026 23:59:32] "POST /predict HTTP/1.1" 200 -
